# 01 · Data preparation and validation

**Question.** Is the data complete and internally consistent before anything is
claimed from it?

This notebook does three things: it loads the panel for the active mode, runs the
validation checks, and prints the cause-field coverage that scopes every later
statement. No figure here — this stage exists so the next five can be trusted.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from team01 import config, io, clean

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)
print(config.summarise())

Team01-FlightDelay-Project v0.2.0 | mode=sample | years=2015-2024 | flights=flights_sample.parquet


## 1.1 Load and validate

`clean.validate` counts rows, duplicates, missingness and threshold coverage. It
raises on a duplicate rate above 0.1% or on more than 5% missing arrival delays,
because either would make the panel unusable rather than merely noisy.

In [2]:
raw = io.load_flights()
checks = clean.validate(raw, f"panel_{config.mode()}")
pd.Series(checks, name="value").to_frame()

,value
stage,panel_sample
rows,47893
duplicate_rows,0
duplicate_rate,0.0
missing_required_cells,961
missing_arr_delay,961
completed_rows,46932
cancelled_rows,833
diverted_rows,131
mean_arr_delay,10.17


## 1.2 The three row populations

Mixing these up is the classic error in this dataset: cause shares describe
*delayed* flights, while means and percentiles describe *completed* flights.

In [3]:
panel = clean.completed(raw)
universe = clean.attribution_universe(raw)

print(f"reported rows          {len(raw):>12,}")
print(f"completed flights      {len(panel):>12,}  ({len(panel)/len(raw):.1%} of reported)")
print(f"attribution universe   {len(universe):>12,}  ({len(universe)/len(panel):.1%} of completed)")

reported rows                47,893
completed flights            46,932  (98.0% of reported)
attribution universe          9,694  (20.7% of completed)


## 1.3 Coverage of the cause fields

BTS publishes the five cause-minute fields only above 15 minutes of arrival delay.
The check is direct: coverage should be 0% below the threshold and about 100% above
it, in every year.

In [4]:
from team01 import attribution

attribution.threshold_audit(raw)

,year,completed,above_threshold,coverage_above,coverage_below
0,2019,16063,3183,1.0,0.0
1,2023,15819,3212,1.0,0.0
2,2024,16011,3299,1.0,0.0


## 1.4 Per-year summary

What a reader needs before any trend line: how many flights, how late, and how
often severe.

In [5]:
clean.summarise_panel(raw)

,year,completed_flights,mean_arr_delay,median_arr_delay,share_delayed_15,share_delayed_60,cancelled_flights
0,2019,15730,10.108,-1.844,0.20235,0.08932,289
1,2023,15512,9.922,-2.112,0.20707,0.08755,261
2,2024,15690,10.478,-1.094,0.21026,0.08821,283
